# M4 / T4.7 消融与特征重要性（C1 / C2 两个受控比较）

- **C1**：$\Delta_{\mathrm{C1}} = \mathrm{LL}(B5a) - \mathrm{LL}(B4)$ —— 只换模型（同信息、同权重），回答「换成机器学习值不值」。B4 预测由 M3 交付（`results/preds/preds_B4.csv`）。
- **C2**：$\Delta_{\mathrm{C2}} = \mathrm{LL}(B5a) - \mathrm{LL}(B5b)$ —— 只换信息（同模型、同权重、同超参），回答「多喂信息值不值」。

两个差的 **90% 区间由 M2 的簇 bootstrap 出**（本 notebook 只算点估计与配对逐场差，不自行下结论）。区间口径：48 块重采样，见 `src/eval/bootstrap.py`（M2）。

In [1]:
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 项目根定位：cwd 可能是仓库根（nbconvert）或 analysis/（Jupyter）。
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.models._common import log_loss, load_blocks, load_panel  # noqa: E402
from src.splits import resolve_split  # noqa: E402
from src.features import FEATURE_COLS, fit_features  # noqa: E402
from src.models.b5_gbm import _fit_hgb, _proba, _weights  # noqa: E402

PREDS = ROOT / "results" / "preds"
FIGS = ROOT / "products" / "figures"
FIGS.mkdir(parents=True, exist_ok=True)

panel = load_panel()
blocks = load_blocks()

In [2]:
def read_preds(name: str) -> pd.DataFrame:
    df = pd.read_csv(PREDS / f"preds_{name}.csv")
    return df.set_index("match_id")[["p_home", "p_draw", "p_away", "y"]]

def paired_delta(a: str, b: str) -> dict:
    """逐场配对差：match_id 内连接后 LL(a) − LL(b)。"""
    pa, pb = read_preds(a), read_preds(b)
    common = pa.index.intersection(pb.index)
    pa, pb = pa.loc[common], pb.loc[common]
    y = pa["y"].to_numpy()
    la = log_loss(y, pa[["p_home", "p_draw", "p_away"]].to_numpy())
    lb = log_loss(y, pb[["p_home", "p_draw", "p_away"]].to_numpy())
    per_match = -np.log(np.clip(pa[["p_home", "p_draw", "p_away"]].to_numpy(), 1e-15, 1)
                        [np.arange(len(y)), y]) \
              + np.log(np.clip(pb[["p_home", "p_draw", "p_away"]].to_numpy(), 1e-15, 1)
                        [np.arange(len(y)), y])
    return {"a": a, "b": b, "n_paired": len(common),
            "LL_a": la, "LL_b": lb, "delta": la - lb,
            "per_match": pd.Series(per_match, index=common)}

In [3]:
# ---- 拟合对象：E0|2023/24 一块，B5b 全特征 + 时间衰减权重（与主跑同口径）----
from src.models._common import load_xi_map  # noqa: E402

block = next(b for b in blocks if b["block_id"] == "E0|2023/24")
train, test = resolve_split(panel, block)
x_train, builder = fit_features(train)

tuning = pd.read_csv(ROOT / "results" / "logs" / "b5_tuning.csv")
row = tuning[tuning["block_id"] == "E0|2023/24"].iloc[0]
print(f"E0|2023/24 调参结果: max_leaf_nodes={int(row['max_leaf_nodes'])} "
      f"learning_rate={float(row['learning_rate'])}")

xi_map = load_xi_map()
xi = xi_map.get("E0|2023/24", 0.0019)
w_train = _weights(train, xi)

clf = _fit_hgb(x_train[FEATURE_COLS], train["y"].to_numpy(), w_train,
               int(row["max_leaf_nodes"]), float(row["learning_rate"]))
print(f"n_train={len(train)}, n_features={len(FEATURE_COLS)}, xi={xi}")

E0|2023/24 调参结果: max_leaf_nodes=7 learning_rate=0.03


n_train=7600, n_features=39, xi=0.0005


In [4]:
# ---- C2：B5a vs B5b（只换信息）----
c2 = paired_delta("B5a", "B5b")
print(f"C2  Δ = LL(B5a) − LL(B5b) = {c2['LL_a']:.5f} − {c2['LL_b']:.5f} "
      f"= {c2['delta']:+.5f} nats（n={c2['n_paired']}）")
print("解释：正 = 加信息带来提升；量级 0.005–0.02 属预期（M4 卡 T4.6）")

# ---- C1：B4 vs B5a（只换模型；B4 由 M3 交付）----
if (PREDS / "preds_B4.csv").exists():
    c1 = paired_delta("B5a", "B4")
    print(f"C1  Δ = LL(B5a) − LL(B4) = {c1['LL_a']:.5f} − {c1['LL_b']:.5f} "
          f"= {c1['delta']:+.5f} nats（n={c1['n_paired']}）")
else:
    c1 = None
    print("preds_B4.csv 尚未交付（M3），C1 待 M3 预测落盘后补算。")
    print("提醒：C1 要求 B4 与 B5a 信息集与样本权重一致（白名单已由 M3 签字）。")

C2  Δ = LL(B5a) − LL(B5b) = 1.00806 − 1.00279 = +0.00527 nats（n=16733）
解释：正 = 加信息带来提升；量级 0.005–0.02 属预期（M4 卡 T4.6）
C1  Δ = LL(B5a) − LL(B4) = 1.00806 − 0.99386 = +0.01421 nats（n=16733）


In [5]:
# sklearn HGB 不暴露树内 gain；用 permutation importance（对验证段逐特征打乱）估算重要性。
from sklearn.inspection import permutation_importance  # noqa: E402

val_season = train["season"].max()
mask_val = (train["season"] == val_season).to_numpy()
x_val, y_val = x_train[FEATURE_COLS][mask_val], train["y"].to_numpy()[mask_val]

r = permutation_importance(clf, x_val, y_val, n_repeats=5, random_state=0,
                           scoring="neg_log_loss", sample_weight=w_train[mask_val])
imp = pd.Series(r.importances_mean, index=FEATURE_COLS).sort_values(ascending=False)
imp = imp[imp > 0].head(15)

fig, ax = plt.subplots(figsize=(7, 5), dpi=300)
ax.barh(np.arange(len(imp))[::-1], imp.values, color="#2c3e50")
ax.set_yticks(np.arange(len(imp))[::-1])
ax.set_yticklabels(imp.index)
ax.set_xlabel("Permutation importance (Δ log loss on held-out last train season)")
ax.set_title("B5b feature importance (E0|2023/24, permutation on inner-val)")
plt.tight_layout()
fig.savefig(FIGS / "fig08_importance.png")
print("已写入 products/figures/fig08_importance.png")
print(imp.to_string())

已写入 products/figures/fig08_importance.png
elo_diff      0.075387
a_shf5        0.019616
h_sha5        0.018700
a_sha5        0.015000
a_opp_elo5    0.013536
a_gf10        0.010468
matchday      0.010260
h_elo         0.008357
h_ga10        0.008120
a_elo         0.008076
h_sotf5       0.007559
h_sota5       0.007302
h_opp_elo5    0.006958
h_shf5        0.006100
a_sotf5       0.006023


## 一句话结论（供 M5 写报告 §4.3 / §5.1）

- 换模型（B4→B5a，同信息同权重）带来约 `ΔC1` nats（区间由 M2 出）。
- 加信息（B5a→B5b，同模型同权重同超参）带来约 `ΔC2` nats（区间由 M2 出）。
- 重要性图里**没有任何赔率类特征**（白名单与全特征表都不含 `PSCH/PSCD/PSCA`）。